In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# =========================================================
# PATHS
# =========================================================
PRED_CSV = Path(r"D:\AI\outputs\runs_sarcasm_improve\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER\predictions.csv")
OUT_DIR = Path(r"D:\AI\outputs\final_submission_package\tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not PRED_CSV.exists():
    raise FileNotFoundError(f"Could not find predictions file:\n{PRED_CSV}")

df = pd.read_csv(PRED_CSV)

required = ["y_true", "p_text", "p_image", "p_emoji", "p_multi"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"predictions.csv missing required columns: {missing}")

y_true = df["y_true"].astype(int).to_numpy()
P = df[["p_text", "p_image", "p_emoji", "p_multi"]].astype(float).to_numpy()

# =========================================================
# HELPERS
# =========================================================
def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, 1.0)
    s = w.sum()
    if s <= 0:
        return np.array([0.25, 0.25, 0.25, 0.25], dtype=float)
    return w / s

def metrics_from_probs(y_true, probs, thr):
    y_hat = (probs >= thr).astype(int)
    return {
        "acc": accuracy_score(y_true, y_hat),
        "macro_f1": f1_score(y_true, y_hat, average="macro", zero_division=0),
        "auc": roc_auc_score(y_true, probs),
    }

def best_threshold_by_macro_f1(y_true, probs, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.951, 0.025), 3)
    best_thr = None
    best_f1 = -1.0
    for thr in thresholds:
        y_hat = (probs >= thr).astype(int)
        f1 = f1_score(y_true, y_hat, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1

def gwo_optimize_weights_fixed_preds(y_true, P, seed=42, n_wolves=20, max_iter=100):
    rng = np.random.default_rng(seed)

    # initialize wolves on simplex
    wolves = rng.random((n_wolves, 4))
    wolves = np.array([normalize_weights(w) for w in wolves])

    def fitness(w):
        probs = P @ w
        thr, best_f1 = best_threshold_by_macro_f1(y_true, probs)
        return best_f1, thr, probs

    best_thr_final = 0.5

    for t in range(max_iter):
        scored = []
        for w in wolves:
            f1, thr, probs = fitness(w)
            scored.append((f1, thr, w.copy()))

        scored = sorted(scored, key=lambda x: x[0], reverse=True)
        alpha_f1, alpha_thr, alpha = scored[0]
        beta = scored[1][2]
        delta = scored[2][2]
        best_thr_final = alpha_thr

        a = 2 - 2 * (t / max_iter)

        new_wolves = [alpha.copy(), beta.copy(), delta.copy()]
        for i in range(3, n_wolves):
            w = wolves[i].copy()

            r1 = rng.random(4)
            r2 = rng.random(4)
            A1 = 2 * a * r1 - a
            C1 = 2 * r2

            r1 = rng.random(4)
            r2 = rng.random(4)
            A2 = 2 * a * r1 - a
            C2 = 2 * r2

            r1 = rng.random(4)
            r2 = rng.random(4)
            A3 = 2 * a * r1 - a
            C3 = 2 * r2

            D_alpha = np.abs(C1 * alpha - w)
            D_beta  = np.abs(C2 * beta  - w)
            D_delta = np.abs(C3 * delta - w)

            X1 = alpha - A1 * D_alpha
            X2 = beta  - A2 * D_beta
            X3 = delta - A3 * D_delta

            w_new = (X1 + X2 + X3) / 3.0
            w_new = normalize_weights(w_new)
            new_wolves.append(w_new)

        wolves = np.array(new_wolves)

    # final alpha
    final_scored = []
    for w in wolves:
        f1, thr, probs = fitness(w)
        final_scored.append((f1, thr, w.copy(), probs.copy()))
    final_scored = sorted(final_scored, key=lambda x: x[0], reverse=True)

    best_f1, best_thr, best_w, best_probs = final_scored[0]
    m = metrics_from_probs(y_true, best_probs, best_thr)

    return {
        "seed": seed,
        "best_thr": float(best_thr),
        "acc": float(m["acc"]),
        "macro_f1": float(m["macro_f1"]),
        "auc": float(m["auc"]),
        "w_text": float(best_w[0]),
        "w_image": float(best_w[1]),
        "w_emoji": float(best_w[2]),
        "w_multi": float(best_w[3]),
    }

# =========================================================
# RUN 5-SEED STABILITY
# =========================================================
seeds = [11, 22, 33, 44, 55]
rows = []

for s in seeds:
    row = gwo_optimize_weights_fixed_preds(y_true, P, seed=s, n_wolves=20, max_iter=100)
    rows.append(row)

seed_df = pd.DataFrame(rows)

summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro-F1", "ROC-AUC", "w_text", "w_image", "w_emoji", "w_multi", "best_thr"],
    "Mean ± Std": [
        f"{seed_df['acc'].mean():.4f} ± {seed_df['acc'].std(ddof=0):.4f}",
        f"{seed_df['macro_f1'].mean():.4f} ± {seed_df['macro_f1'].std(ddof=0):.4f}",
        f"{seed_df['auc'].mean():.4f} ± {seed_df['auc'].std(ddof=0):.4f}",
        f"{seed_df['w_text'].mean():.4f} ± {seed_df['w_text'].std(ddof=0):.4f}",
        f"{seed_df['w_image'].mean():.4f} ± {seed_df['w_image'].std(ddof=0):.4f}",
        f"{seed_df['w_emoji'].mean():.4f} ± {seed_df['w_emoji'].std(ddof=0):.4f}",
        f"{seed_df['w_multi'].mean():.4f} ± {seed_df['w_multi'].std(ddof=0):.4f}",
        f"{seed_df['best_thr'].mean():.4f} ± {seed_df['best_thr'].std(ddof=0):.4f}",
    ]
})

# Save
seed_csv = OUT_DIR / "table3_seedwise_gwo_stability.csv"
summary_csv = OUT_DIR / "table3_final.csv"
seed_df.to_csv(seed_csv, index=False)
summary.to_csv(summary_csv, index=False)

# Optional note
note_path = OUT_DIR / "table3_method_note.txt"
note = (
    "Table 3 reports a five-seed stability analysis of post-hoc GWO fusion on fixed in-domain branch predictions "
    "from the final v4 run. This is not a full multi-seed end-to-end retraining analysis."
)
note_path.write_text(note, encoding="utf-8")

print("Saved:", seed_csv)
print("Saved:", summary_csv)
print("Saved:", note_path)

display(seed_df)
display(summary)

Saved: D:\AI\outputs\final_submission_package\tables\table3_seedwise_gwo_stability.csv
Saved: D:\AI\outputs\final_submission_package\tables\table3_final.csv
Saved: D:\AI\outputs\final_submission_package\tables\table3_method_note.txt


,seed,best_thr,acc,macro_f1,auc,w_text,w_image,w_emoji,w_multi
0,11,0.5,0.654909,0.516676,0.502638,0.493544,0.164265,0.154768,0.187423
1,22,0.5,0.651096,0.515465,0.502675,0.430840,0.261087,0.044353,0.263720
2,33,0.5,0.654909,0.516676,0.502707,0.500792,0.155676,0.160043,0.183490
3,44,0.5,0.651096,0.516816,0.501929,0.098869,0.527683,0.109889,0.263559
4,55,0.5,0.653003,0.515387,0.502464,0.521624,0.121551,0.222286,0.134540


,Metric,Mean ± Std
0,Accuracy,0.6530 ± 0.0017
1,Macro-F1,0.5162 ± 0.0006
2,ROC-AUC,0.5025 ± 0.0003
3,w_text,0.4091 ± 0.1581
4,w_image,0.2461 ± 0.1483
5,w_emoji,0.1383 ± 0.0590
6,w_multi,0.2065 ± 0.0502
7,best_thr,0.5000 ± 0.0000
